In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report
df = pd.read_csv('spam.csv',encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [2]:
df =df[['v1','v2']]
df.columns = ['label','message']
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
print("Shape:",df.shape)
print("\nMissing Values:\n",df.isnull().sum())
print("\nClass distribution;\n",df['label'].value_counts())
print("\nClass distribution (%):",df['label'].value_counts(normalize=True)*100)

Shape: (5572, 2)

Missing Values:
 label      0
message    0
dtype: int64

Class distribution;
 label
ham     4825
spam     747
Name: count, dtype: int64

Class distribution (%): label
ham     86.593683
spam    13.406317
Name: proportion, dtype: float64


In [4]:
import re
import nltk
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]',"",text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)
df['clean_message'] = df['message'].apply(clean_text)
df[['message','clean_message']].head()

,message,clean_message
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df['clean_message'])
y = df['label']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print("TF-IDF matrix shape:",X_train.shape[0])
print("Testing samples:",X_test.shape[0])
print("Full TF-IDF matrix shape:",X.shape)


TF-IDF matrix shape: 4457
Testing samples: 1115
Full TF-IDF matrix shape: (5572, 8389)


## What is TF-IDF?: 
TF-IDF just means"how important is this word in this message,compared to all messages?"
if a word shows up a lot in one message but barely anywhere else (like "free" or "winner"),it gets a high score.if it's a word that shows up everywhere anyway(like "the" or "and" ),it gets a low score since it doesn't really tell us anything useful. This helps th model pay attention to words that actually matter for spotting spam,instead of common words that show up in every message.

In [6]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report
nb_model = MultinomialNB()
nb_model.fit(X_train,y_train)
nb_pred = nb_model.predict(X_test)
print("=== Naive Bayes Results ===")
print("Accuracy:",accuracy_score(y_test,nb_pred))
print("\nClassification Report:\n",classification_report(y_test,nb_pred))
print("\nConfusion Matrix:\n",confusion_matrix(y_test,nb_pred))

=== Naive Bayes Results ===
Accuracy: 0.9641255605381166

Classification Report:
               precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.73      0.84       149

    accuracy                           0.96      1115
   macro avg       0.98      0.87      0.91      1115
weighted avg       0.97      0.96      0.96      1115


Confusion Matrix:
 [[966   0]
 [ 40 109]]


In [7]:
from sklearn.linear_model import LogisticRegression
ir_model = LogisticRegression(max_iter=1000)
ir_model.fit(X_train,y_train)
ir_pred = ir_model.predict(X_test)
print("=== Logistic Regression Result ===")
print("Accuracy:",accuracy_score(y_test,ir_pred))
print("\nClassification Report:\n",classification_report(y_test,ir_pred))
print("\nConfusion Matrix:\n",confusion_matrix(y_test,ir_pred))

=== Logistic Regression Result ===
Accuracy: 0.95695067264574

Classification Report:
               precision    recall  f1-score   support

         ham       0.95      1.00      0.98       966
        spam       0.99      0.68      0.81       149

    accuracy                           0.96      1115
   macro avg       0.97      0.84      0.89      1115
weighted avg       0.96      0.96      0.95      1115


Confusion Matrix:
 [[965   1]
 [ 47 102]]


## Why does recall matter here?:
Both models are really good atnot falsely calling a normal message "spam"(precision is 0.99-1.00).but their recall for spam is lower(0.68-073),which means they're missing about to a third actual spam messages.
Is that okay?kind of,yeah.Missing some spam is way less annoying than accidentally blocking a message from ypur mom or your boss.So a model that's cautious about calling things "spam" makes sense for real-world use.If we wanted to catch more spam,We could tweak the model,but we'd risk more false alarms too.

## Conclusion:
I trained two models for this -Naive Bayes and Logistic Regression -and compared how well they caught spam.
|Model|Accuracy|
|---|---|
|Naive Bayes|96.4%|
|Logistic Regression|95.7%|
Naive Bayes came out on top,so i'm going with that one.it's not flawless -it still lets a few spam messages slip through --but overall it does a pretty solid job telling spam from real messages.